# Apply Text Preprocessing Methods (Plain Python)

## Import packages

In [14]:
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_colwidth", 200)

## Helper functions

In [15]:
def load_dataframe(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)
    if suffix == ".json":
        return pd.read_json(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix in {".pkl", ".pickle"}:
        return pd.read_pickle(path)

    raise ValueError(f"Unsupported input file type: {suffix}")


def save_dataframe(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        df.to_csv(path, index=False)
        return
    if suffix in {".parquet", ".pq"}:
        df.to_parquet(path, index=False)
        return
    if suffix == ".jsonl":
        df.to_json(path, orient="records", lines=True)
        return
    if suffix == ".json":
        df.to_json(path, orient="records")
        return
    if suffix in {".pkl", ".pickle"}:
        df.to_pickle(path)
        return

    raise ValueError(f"Unsupported output file type: {suffix}")


def as_text_series(series):
    return series.fillna("").astype(str)


def as_nullable_text_series(series):
    return series.astype("string")


def normalize_columns(columns):
    if columns is None:
        return []
    if isinstance(columns, str):
        return [col.strip() for col in columns.split(",") if col.strip()]
    return list(columns)


def require_columns(df, columns):
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")


def load_eval_subsets(paths):
    dataframes = [load_dataframe(path) for path in paths]
    if not dataframes:
        raise ValueError("Provide at least one evaluation subset path")

    return (
        pd.concat(dataframes, ignore_index=True)
        .drop_duplicates()
        .reset_index(drop=True)
    )


def build_preprocessed_df(df, id_cols, target_col, transformed_target):
    id_cols = normalize_columns(id_cols)
    require_columns(df, id_cols + [target_col])

    output_df = df[id_cols].copy()
    output_df[f"{target_col}_original"] = df[target_col]
    output_df[target_col] = transformed_target
    return output_df


def remove_common_punctuation(series):
    cleaned = as_nullable_text_series(series)
    for punctuation in [",", '"', ":"]:
        cleaned = cleaned.str.replace(punctuation, "", regex=False)
    return cleaned

## Set parameters

In [ ]:
# Replace these paths and column names for your local data.
eval_subset_paths = [
    Path("data/longest_unique_start_records.csv"),
    Path("data/random_records_by_unit.csv"),
]

target_col = "text"
id_cols = ["encounter_id", "lab_eventnumber"]
context_col = "category"

eval_subset_path = Path("data/model_inputs/all_eval_data.csv")
drop_punctuation_path = Path("data/model_inputs/drop_punctuation.csv")
uppercase_path = Path("data/model_inputs/uppercase.csv")
lowercase_path = Path("data/model_inputs/lowercase.csv")
context_col_path = Path("data/model_inputs/context_col.csv")

In [17]:
# # Replace these paths and column names for your local data.
# eval_subset_paths = [
#     Path("data/longest_unique_start_records.csv"),
#     Path("data/random_records_by_unit.csv"),
# ]

# target_col = "text"
# id_cols = ["hc_id", "encounter_id", "lab_eventnumber"]
# context_col = None

# eval_subset_path = Path("data/all_eval_data.csv")
# drop_punctuation_path = Path("data/drop_punctuation.csv")
# uppercase_path = Path("data/uppercase.csv")
# lowercase_path = Path("data/lowercase.csv")
# context_col_path = Path("data/context_col.csv")


## Combine all evaluation data as one file

In [ ]:
eval_subset_df = load_eval_subsets(eval_subset_paths)
require_columns(eval_subset_df, normalize_columns(id_cols) + [target_col])

display(eval_subset_df.head())
print(f"Loaded {len(eval_subset_df):,} unique evaluation subset rows")

save_dataframe(eval_subset_df, eval_subset_path)
print(f"Saved {len(eval_subset_df):,} rows to {eval_subset_path.resolve()}")

## Remove Common Punctuation

In [ ]:
drop_punctuation_df = build_preprocessed_df(
    eval_subset_df,
    id_cols=id_cols,
    target_col=target_col,
    transformed_target=remove_common_punctuation(eval_subset_df[target_col]),
)

display(drop_punctuation_df.head())
save_dataframe(drop_punctuation_df, drop_punctuation_path)
print(
    f"Saved {len(drop_punctuation_df):,} rows to {drop_punctuation_path.resolve()}"
)

## All Upper Case

In [ ]:
uppercase_df = build_preprocessed_df(
    eval_subset_df,
    id_cols=id_cols,
    target_col=target_col,
    transformed_target=as_nullable_text_series(
        eval_subset_df[target_col]
    ).str.upper(),
)

display(uppercase_df.head())
save_dataframe(uppercase_df, uppercase_path)
print(f"Saved {len(uppercase_df):,} rows to {uppercase_path.resolve()}")

## All Lower Case

In [ ]:
lowercase_df = build_preprocessed_df(
    eval_subset_df,
    id_cols=id_cols,
    target_col=target_col,
    transformed_target=as_nullable_text_series(
        eval_subset_df[target_col]
    ).str.lower(),
)

display(lowercase_df.head())
save_dataframe(lowercase_df, lowercase_path)
print(f"Saved {len(lowercase_df):,} rows to {lowercase_path.resolve()}")

## Add Context from Another Column

In [ ]:
# Set context_col to a column name before running this cell.
if context_col is None:
    print("Set context_col to create the context-augmented output.")
else:
    require_columns(eval_subset_df, [context_col])
    context_target = (
        as_text_series(eval_subset_df[context_col])
        + " "
        + as_text_series(eval_subset_df[target_col])
    )
    context_col_df = build_preprocessed_df(
        eval_subset_df,
        id_cols=id_cols,
        target_col=target_col,
        transformed_target=context_target,
    )

    display(context_col_df.head())
    save_dataframe(context_col_df, context_col_path)
    print(
        f"Saved {len(context_col_df):,} rows to {context_col_path.resolve()}"
    )

# Create deny list

Can be done in the 2.x Run Presidio notebooks. Also see Example 1 here: https://presidio.dataprivacystack.org/samples/python/customizing_presidio_analyzer/